In [ ]:
import pandas as pd
import numpy as np
import json

In [ ]:
def hist_to_json(x, bins=20):
    counts, edges = np.histogram(x, bins=bins)
    return json.dumps([
        {
            "bin_start": float(start),
            "bin_end": float(end),
            "count": int(count),
        }
        for count, start, end in zip(counts, edges[:-1], edges[1:])
    ])

In [ ]:
dwi = pd.read_pickle(snakemake.input.dwi)
ihmt = pd.read_pickle(snakemake.input.ihmt)
mp2rage = pd.read_pickle(snakemake.input.mp2rage)
qMT = pd.read_pickle(snakemake.input.qMT)
qsm = pd.read_pickle(snakemake.input.QSM)

In [ ]:
contrast_list = [
    'fa', 'md', 'kfa', 'mkt',
    'cosmod_ihMTR', 'freqalt_ihMTR',
    'R1map_b1corr', 'R1map',
    'MEGRE_Chimap', 'MEGRE_R2starmap'
]
acq_list = [
    'DWIepse2',
    'ihMTMCsag16isop3',
    'mp2ragewipsag1isoc6cs7DL6820us_rec-DL',
    'sag1isoc9'
]
df_list = [dwi, ihmt, mp2rage, qMT, qsm]
for i, df in enumerate(df_list):
    df_list[i] = df_list[i].query('contrast in @contrast_list and acquisition in @acq_list and region == "wm90percent" and outliers_removed == True')
    df_list[i]["hist"] = df_list[i]["data"].apply(hist_to_json)
    df_list[i] = df_list[i].drop("data", axis=1)

In [ ]:
df = pd.concat(df for df in df_list)

In [ ]:
df

In [ ]:
df.to_csv(snakemake.output[0])